In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from OA_utils.data_utils import *
import random
import pickle

Load data from file

In [3]:
data_dir      = 'C:\\Users\\bakel\\Desktop\\GRFMuscleModel\\data\\'
filename      = 'Ulrich_resampled_segments.pkl'   # <-- change to your file
output_prefix = 'Ulrich_jrf_test_only'                      # <-- prefix for saved .npz files

SKIP_SUBJECTS = {'time_resampled', 'OA11'}  # subjects to exclude

# Set True to package the entire dataset as a test set only (no train/val split).
# Useful for cross-testing models trained on a different dataset.
TEST_ONLY = True

with open(data_dir + filename, 'rb') as f:
    segs = pickle.load(f)

print('Loaded', filename)

Loaded Ulrich_resampled_segments.pkl


Visualize amount of data present for each subject

In [ ]:
def count_total_segments(split_dict, key='grf_y'):
    return sum(len(split_dict[subj][key]) for subj in split_dict)

subjects   = [s for s in segs.keys() if s not in SKIP_SUBJECTS]
total_segs = 0
for subj in subjects:
    n = len(segs[subj]['grf_y'])
    total_segs += n
    print(f'{subj}: {n} segments')
print(total_segs, 'segments total')

## Split
If `TEST_ONLY = True`, the entire dataset becomes the test set and the seed search is skipped.

In [ ]:
if TEST_ONLY:
    train_dict = {}
    val_dict   = {}
    test_dict  = {s: segs[s] for s in subjects}
    print(f'TEST_ONLY mode — all {total_segs} segments → test set')

else:
    best_seed = None
    best_err  = float('inf')
    best_counts = []
    target_train, target_val, target_test = 0.8, 0.1, 0.1

    for seed in range(40):
        random.seed(seed)
        shuf = subjects.copy()
        random.shuffle(shuf)

        N         = len(shuf)
        train_end = int(N * 0.8)
        val_end   = int(N * 0.9)

        td = {s: segs[s] for s in shuf[:train_end]}
        vd = {s: segs[s] for s in shuf[train_end:val_end]}
        ed = {s: segs[s] for s in shuf[val_end:]}

        tc = count_total_segments(td)
        vc = count_total_segments(vd)
        ec = count_total_segments(ed)

        err = (abs(tc/total_segs - target_train) +
               abs(vc/total_segs - target_val)   +
               abs(ec/total_segs - target_test))

        if err < best_err:
            best_err    = err
            best_seed   = seed
            best_counts = (tc, vc, ec)

        print('Best seed:', best_seed,
              ' counts:', best_counts,
              ' error:', round(best_err, 6))

    # Rebuild with best seed
    random.seed(best_seed)
    shuf = subjects.copy()
    random.shuffle(shuf)
    N         = len(shuf)
    train_end = int(N * 0.8)
    val_end   = int(N * 0.9)

    train_dict = {s: segs[s] for s in shuf[:train_end]}
    val_dict   = {s: segs[s] for s in shuf[train_end:val_end]}
    test_dict  = {s: segs[s] for s in shuf[val_end:]}

    print(f'\nBest seed: {best_seed}   ratios: {best_counts[0]/total_segs:.2f} / {best_counts[1]/total_segs:.2f} / {best_counts[2]/total_segs:.2f}')
    print('Train:', list(train_dict.keys()), '->', count_total_segments(train_dict), 'segs')
    print('Val:  ', list(val_dict.keys()),   '->', count_total_segments(val_dict),   'segs')
    print('Test: ', list(test_dict.keys()),  '->', count_total_segments(test_dict),  'segs')

In [9]:
print(segs['Subject1'].keys())

dict_keys(['grf_x', 'grf_y', 'grf_z', 'cop_x', 'cop_y', 'cop_z', 'tibpost', 'tibant', 'edl', 'ehl', 'fdl', 'fhl', 'gaslat', 'gasmed', 'soleus', 'perbrev', 'perlong', 'achilles', 'subtalar_calc_fx', 'subtalar_calc_fy', 'subtalar_calc_fz', 'ankle_talus_fx', 'ankle_talus_fy', 'ankle_talus_fz', 'knee_tibia_fx', 'knee_tibia_fy', 'knee_tibia_fz'])


In [11]:
def get_subject_ranges(split_dict):
    ranges = {}
    start_idx = 0
    for subj, data in split_dict.items():
        n = len(data['grf_x'])
        end_idx = start_idx + n
        ranges[subj] = (start_idx, end_idx)
        start_idx = end_idx
    return ranges

print('Test subject ranges:', get_subject_ranges(test_dict))

Test subject ranges: {'Subject5': (0, 1482)}


In [ ]:
INPUT_KEYS  = ['grf_x', 'grf_y', 'grf_z', 'cop_x', 'cop_y', 'cop_z']
OUTPUT_KEYS = [
    'tibpost', 'tibant', 'edl', 'ehl', 'fdl', 'fhl',
    'gaslat', 'gasmed', 'soleus', 'perbrev', 'perlong',
    'achilles', 'subtalar_calc_fx', 'subtalar_calc_fy',
    'subtalar_calc_fz', 'ankle_talus_fx', 'ankle_talus_fy',
    'ankle_talus_fz', 'knee_tibia_fx', 'knee_tibia_fy', 'knee_tibia_fz',
]
ALL_KEYS  = INPUT_KEYS + OUTPUT_KEYS
N_INPUTS  = len(INPUT_KEYS)
N_OUTPUTS = len(OUTPUT_KEYS)

print(f'Inputs  ({N_INPUTS}):  {INPUT_KEYS}')
print(f'Outputs ({N_OUTPUTS}): {OUTPUT_KEYS}')

## Signal definitions — update here when adding new inputs/outputs

In [14]:
np.savez(data_dir + f'{output_prefix}_train_data.npz', X_train=X_train, y_train=y_train)
np.savez(data_dir + f'{output_prefix}_val_data.npz',   X_val=X_val,     y_val=y_val)
np.savez(data_dir + f'{output_prefix}_test_data.npz',  X_test=X_test,   y_test=y_test)